# Day 2: Strings and files (Solutions)

**Python for Everybody:** Course 2 *Python Data Structures*, chapters 6 (Strings) and 7 (Files). Includes the famous `mbox-short.txt` exercises.

**Why it matters for the JD:** Reporting and data-processing jobs mean parsing text: log lines, emails, CSV-ish files. Interviewers love string-manipulation questions and `with open(...)`.

**Time:** about 75 minutes · 7 exercises

These are reference solutions. Try the exercises notebook first, then compare. Every cell here passes its check.

In [ ]:
# Run this cell first. It gives you the check() helper used below.
_results = {}

def check(name, test):
    """Run a test function and print a pass/fail line with a hint."""
    try:
        test()
    except AssertionError as e:
        _results[name] = False
        print(f"❌ {name}: {e}")
    except Exception as e:
        _results[name] = False
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        _results[name] = True
        print(f"✅ {name}")

def raises(exc, fn, *args, **kwargs):
    """True if fn(*args, **kwargs) raises exc."""
    try:
        fn(*args, **kwargs)
    except exc:
        return True
    except Exception:
        return False
    return False

TOTAL = 7

def score():
    done = sum(_results.values())
    print(f"Score: {done}/{TOTAL}  " + ("🎉 all green!" if done == TOTAL else "keep going, re-run the cells you missed"))


## Exercise 1: clean_name(raw): split, join, title  `warm-up`

Remove leading/trailing spaces, collapse repeated spaces to one, and Title Case the result.

`clean_name("  jOHN   dOE ")` -> `"John Doe"`

In [ ]:
def clean_name(raw):
    return " ".join(raw.split()).title()

In [ ]:
def _test():
    assert clean_name("  jOHN   dOE ") == "John Doe"
    assert clean_name("ada") == "Ada"
    assert clean_name("   ") == "", "only spaces gives an empty string"

check('1. clean_name(raw): split, join, title', _test)

## Exercise 2: is_palindrome(s): ignore case and punctuation  `easy`

Return `True` if the text reads the same backwards, looking only at letters and digits and ignoring case.

`is_palindrome("A man, a plan, a canal: Panama")` -> `True`

In [ ]:
def is_palindrome(s):
    cleaned = [c.lower() for c in s if c.isalnum()]
    return cleaned == cleaned[::-1]

In [ ]:
def _test():
    assert is_palindrome("A man, a plan, a canal: Panama") is True
    assert is_palindrome("hello") is False
    assert is_palindrome("") is True, "an empty string is a palindrome"
    assert is_palindrome("No 'x' in Nixon") is True

check('2. is_palindrome(s): ignore case and punctuation', _test)

## Exercise 3: extract_domain(email)  `easy`

Return the lower-cased domain of an email address, or `None` if it isn't a valid-looking address (must contain exactly one `@`, with text on both sides).

`extract_domain("alice@Example.COM")` -> `"example.com"`

In [ ]:
def extract_domain(email):
    if email.count("@") != 1:
        return None
    user, domain = email.split("@")
    if not user or not domain:
        return None
    return domain.lower()

In [ ]:
def _test():
    assert extract_domain("alice@Example.COM") == "example.com"
    assert extract_domain("bad") is None, "no @"
    assert extract_domain("@x.com") is None, "nothing before the @"
    assert extract_domain("a@") is None, "nothing after the @"
    assert extract_domain("a@b@c") is None, "two @ signs"

check('3. extract_domain(email)', _test)

## Exercise 4: hour_of(from_line): the py4e mbox classic  `medium`

Lines in a mail archive look like:

`From stephen.marquard@uct.ac.za Sat Jan  5 09:14:16 2008`

Return the **hour** as a string (`"09"`). Use `split()` and then split the time on `":"`.

In [ ]:
def hour_of(from_line):
    words = from_line.split()
    return words[5].split(":")[0]

In [ ]:
def _test():
    assert hour_of("From stephen.marquard@uct.ac.za Sat Jan  5 09:14:16 2008") == "09"
    assert hour_of("From a@b.c Mon Feb 12 23:59:59 2008") == "23"

check('4. hour_of(from_line): the py4e mbox classic', _test)

## Exercise 5: save_lines / load_lines: files and `with`  `medium`

Write two functions using `with open(...)`:
- `save_lines(path, lines)` writes each string on its own line.
- `load_lines(path)` returns the lines as a list **without** the trailing newline characters.

Saving then loading must give back the same list.

In [ ]:
def save_lines(path, lines):
    with open(path, "w") as f:
        for line in lines:
            f.write(line + "\n")


def load_lines(path):
    with open(path) as f:
        return [line.rstrip("\n") for line in f]

In [ ]:
def _test():
    import tempfile, os
    with tempfile.TemporaryDirectory() as folder:
        path = os.path.join(folder, "demo.txt")
        save_lines(path, ["a", "b c", ""])
        with open(path) as f:
            assert f.read() == "a\nb c\n\n", "each line should end with a newline when saved"
        assert load_lines(path) == ["a", "b c", ""], "load_lines should strip the newlines"

check('5. save_lines / load_lines: files and `with`', _test)

## Exercise 6: file_stats(path): lines, words, characters  `medium`

Return `{"lines": ..., "words": ..., "chars": ...}` for a text file. `chars` counts every character including newlines. A missing file should raise `FileNotFoundError` (just let `open` do it).

In [ ]:
def file_stats(path):
    with open(path) as f:
        text = f.read()
    return {"lines": len(text.splitlines()), "words": len(text.split()), "chars": len(text)}

In [ ]:
def _test():
    import tempfile, os
    with tempfile.TemporaryDirectory() as folder:
        path = os.path.join(folder, "a.txt")
        with open(path, "w") as f:
            f.write("hello world\nfoo\n")
        assert file_stats(path) == {"lines": 2, "words": 3, "chars": 16}
        empty = os.path.join(folder, "empty.txt")
        open(empty, "w").close()
        assert file_stats(empty) == {"lines": 0, "words": 0, "chars": 0}
        assert raises(FileNotFoundError, file_stats, os.path.join(folder, "nope.txt"))

check('6. file_stats(path): lines, words, characters', _test)

## Exercise 7: Interview: hour_histogram(path) (mbox style)  `interview`

Read a file line by line. Only lines that start with `"From "` (with a space, so `From:` lines don't count) have a timestamp. Count how many messages arrived in each hour and return a **list of `(hour, count)` tuples sorted by hour**. Reuse `hour_of` from exercise 4.

This is the real Dr. Chuck exercise, and a favourite 'process a log file' interview task.

In [ ]:
def hour_histogram(path):
    counts = {}
    with open(path) as f:
        for line in f:
            if not line.startswith("From "):
                continue
            hour = hour_of(line)
            counts[hour] = counts.get(hour, 0) + 1
    return sorted(counts.items())

In [ ]:
def _test():
    import tempfile, os
    lines = [
        "From a@x.com Sat Jan  5 09:14:16 2008\n",
        "From: a@x.com\n",
        "Subject: hello\n",
        "From b@y.org Sun Jan  6 09:30:00 2008\n",
        "From c@z.net Mon Jan  7 17:01:02 2008\n",
    ]
    with tempfile.TemporaryDirectory() as folder:
        path = os.path.join(folder, "mbox.txt")
        with open(path, "w") as f:
            f.writelines(lines)
        assert hour_histogram(path) == [("09", 2), ("17", 1)], "expected [('09', 2), ('17', 1)]. Ignore the 'From:' line."

check('7. Interview: hour_histogram(path) (mbox style)', _test)

## Your score

In [ ]:
score()

## Say it out loud (interview questions)

Answer each one aloud in under 60 seconds, then compare.

**Strings are immutable. What does that mean in practice?**

Methods like `upper()`, `strip()` and `replace()` return a new string and never change the original. You must assign the result (`s = s.strip()`).

**Why use `with open(...)` instead of `open()` and `close()`?**

The `with` block guarantees the file is closed even if an exception happens, so you don't leak file handles. It works through the context-manager protocol (`__enter__`/`__exit__`).

**How do you read a very large file without loading it all into memory?**

Iterate over the file object (`for line in f`). It yields one line at a time lazily, so memory use stays flat.

**How do you reverse a string?**

`s[::-1]` (slice with step -1). `''.join(reversed(s))` also works. Strings have no `.reverse()` method because they are immutable.

**`split()` vs `split(' ')`?**

`split()` with no argument splits on any run of whitespace and drops empty strings. `split(' ')` splits on every single space, so repeated spaces give empty strings.

